In [1]:
import pandas as pd
import numpy as np
from sage.databases.cremona import CremonaDatabase
from sage.schemes.elliptic_curves.ec_database import elliptic_curves
from sage.all import QQ, EllipticCurve
from tqdm import tqdm
import multiprocessing as mp
import os

# ========================= CONFIG =========================
OUTPUT_CSV = "acsc_cremona.csv"
NUM_CURVES_PER_RANK = 38042
NUM_CORES = 6
CHUNK_SIZE = 1000          # Smaller chunks = less memory pressure
RESTART_FROM = 1

# ======================================================

db = CremonaDatabase()
print(f"Using MiniCremonaDatabase — largest conductor = {db.largest_conductor():,}")

# ====================== WORKER FUNCTIONS ======================

def get_rank_worker(N):
    """Fast worker: only get rank and basic info (always reliable in mini-db)"""
    rows = []
    try:
        class_labels = db.curves(N)
        for cls in class_labels:
            label = f"{N}{cls}" if isinstance(cls, str) else f"{N}{cls[0]}"
            try:
                # Use low-level access for rank (very stable)
                rank = db.allcurves(N)[cls][1] if hasattr(db, 'allcurves') else db.elliptic_curve(label).rank()
                row = {
                    "label": label,
                    "conductor": N,
                    "delta": None,           # will fill later if possible
                    "rank": int(rank),
                    "regulator": None,
                    "real_period": None,
                    "tamagawa": None,
                    "sha": None,
                    "torsion": None,
                    "weierstrass_a_invariants": None,
                    "j_invariant": None,
                }
                rows.append(row)
            except Exception as e:
                print(f"Error in get_rank_worker({N}): {e}")

    except Exception as e:
        print(f"Error in get_rank_worker({N}): {e}")

    return rows
    
results = []

for r in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]:
    print(f"\n Rank {r} — fetching Cremona labels...")
    labels = elliptic_curves.rank(rank=r, n=NUM_CURVES_PER_RANK, labels=True)
    
    print(f"   Processing {len(labels)} curves in parallel...")

def get_bsd_worker(label):
    """Separate worker for heavy BSD invariants (omega, regulator, etc.)"""
    try:
        E = db.elliptic_curve(label)
        delta_val = int(E.discriminant())
        try:
            reg_val = float(E.regulator()) if E.rank() > 0 else 1.0
        except:
            reg_val = 1.0
            
        # Safe extraction
        try:
            omega_val = float(E.omega())
        except:
            omega_val = None
            
        try:
            reg_val = float(E.regulator()) if E.rank() > 0 else 1.0
        except:
            reg_val = 1.0
            
        try:
            tam_val = int(E.tamagawa_product())
        except:
            tam_val = None
            
        try:
            sha_val = float(E.sha())
        except:
            sha_val = None
            
        try:
            # Torsion is critical for your denominator weighting
            tors_val = int(E.torsion_order())
        except:
            tors_val = 1 # Default to 1 to avoid div by zero
       
        try:
            a_invs = E.a_invariants()
        except:
            a_invs = None

        try:
            j_inv = E.j_invariant()
        except:
            j_inv = None
    
        return {
            "label": label,
            "delta": delta_val,
            "regulator": reg_val,
            "torsion": tors_val,
            "real_period": omega_val,
            "tamagawa": tam_val,
            "sha": sha_val,
            "conductor": int(E.conductor()),
            "rank": int(E.rank()),
            "weierstrass_a_invariants": a_invs,
            "j_invariant": j_inv,
        }

    except:
        return None

# ====================== MAIN PARALLEL PIPELINE ======================

print(f"Starting parallel extraction with {NUM_CORES} cores...")

all_rows = []
conductors = list(range(RESTART_FROM, db.largest_conductor() + 1))

for i in tqdm(range(0, len(conductors), CHUNK_SIZE), desc="Rank phase (fast)"):
    chunk = conductors[i:i + CHUNK_SIZE]
    with mp.Pool(processes=NUM_CORES) as pool:
        chunk_results = pool.map(get_rank_worker, chunk)
    for res in chunk_results:
        all_rows.extend(res)

print(f"Rank phase complete: {len(all_rows):,} basic entries")

# Second pass: fill BSD invariants in parallel (only for labels we have)
labels_to_process = [row["label"] for row in all_rows if row["rank"] is not None]

print(f"Starting BSD invariants phase for {len(labels_to_process):,} curves...")

bsd_results = []
with mp.Pool(processes=NUM_CORES) as pool:
    for result in tqdm(pool.imap_unordered(get_bsd_worker, labels_to_process), total=len(labels_to_process), desc="BSD phase"):
        if result:
            bsd_results.append(result)

# Merge the two datasets
rank_df = pd.DataFrame(all_rows)
bsd_df = pd.DataFrame(bsd_results)

if not bsd_df.empty:
    final_df = pd.merge(rank_df, bsd_df, on="label", how="left", suffixes=("", "_bsd"))
   
    # Prefer BSD-filled values
    for col in ["delta", "regulator", "real_period", "tamagawa", "sha", "torsion", "weierstrass_a_invariants", "j_invariant"]:
        if col + "_bsd" in final_df.columns:
            final_df[col] = final_df[col + "_bsd"].combine_first(final_df[col])
            final_df.drop(columns=[col + "_bsd"], inplace=True)
else:
    final_df = rank_df

# ACSC derived columns
final_df["log_abs_delta"] = final_df["delta"].abs().apply(lambda x: float(x)**0.5 if pd.notna(x) and x != 0 else 0)
final_df['log_conductor'] = np.log10(final_df['conductor'] + 1)
final_df['bsd_weight'] = final_df['regulator'] / (final_df['torsion']**2)

# Create a robust scale for the Alpha Complex (0.1 to 5.0 range)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0.1, 5.0))
final_df['tda_weight'] = scaler.fit_transform(np.log1p(final_df[['bsd_weight']]))

final_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n Final ACSC dataset saved: {OUTPUT_CSV}")
print(f"   Total curves: {len(final_df):,}")
print(f"   real_period (omega) computed where available")
print("   Rank is fully populated (fast path)")

print("Ready for ACSC testing!")

Using MiniCremonaDatabase — largest conductor = 9,999

 Rank 0 — fetching Cremona labels...
   Processing 30427 curves in parallel...

 Rank 1 — fetching Cremona labels...
   Processing 31871 curves in parallel...

 Rank 2 — fetching Cremona labels...
   Processing 2388 curves in parallel...

 Rank 3 — fetching Cremona labels...
   Processing 836 curves in parallel...

 Rank 4 — fetching Cremona labels...
   Processing 1 curves in parallel...

 Rank 5 — fetching Cremona labels...
   Processing 0 curves in parallel...

 Rank 6 — fetching Cremona labels...
   Processing 0 curves in parallel...

 Rank 7 — fetching Cremona labels...
   Processing 0 curves in parallel...

 Rank 8 — fetching Cremona labels...
   Processing 0 curves in parallel...

 Rank 9 — fetching Cremona labels...
   Processing 0 curves in parallel...

 Rank 10 — fetching Cremona labels...
   Processing 0 curves in parallel...

 Rank 11 — fetching Cremona labels...
   Processing 0 curves in parallel...

 Rank 12 — fetchin

Rank phase (fast):   0%|                                                                                              | 0/10 [00:00<?, ?it/s]

Error in get_rank_worker(169): database disk image is malformedError in get_rank_worker(43): database disk image is malformedError in get_rank_worker(85): database disk image is malformedError in get_rank_worker(211): file is not a databaseError in get_rank_worker(3): file is not a database




Error in get_rank_worker(44): database disk image is malformedError in get_rank_worker(170): database disk image is malformedError in get_rank_worker(86): database disk image is malformedError in get_rank_worker(212): database disk image is malformedError in get_rank_worker(11): database disk image is malformed



Error in get_rank_worker(45): database disk image is malformedError in get_rank_worker(171): database disk image is malformed
Error in get_rank_worker(87): database disk image is malformedError in get_rank_worker(213): database disk image is malformed



Error in get_rank_worker(14): database disk image is malformedError in get_rank_worker(46): database disk image is malformedError in 

Rank phase (fast):  10%|████████▌                                                                             | 1/10 [00:00<00:03,  2.58it/s]

Error in get_rank_worker(1001): database disk image is malformedError in get_rank_worker(1043): database disk image is malformedError in get_rank_worker(1211): database disk image is malformedError in get_rank_worker(1169): database disk image is malformed



Error in get_rank_worker(1002): database disk image is malformedError in get_rank_worker(1044): database disk image is malformedError in get_rank_worker(1212): database disk image is malformed
Error in get_rank_worker(1170): database disk image is malformed

Error in get_rank_worker(1003): database disk image is malformed
Error in get_rank_worker(1045): database disk image is malformedError in get_rank_worker(1215): database disk image is malformed
Error in get_rank_worker(1171): database disk image is malformed

Error in get_rank_worker(1005): database disk image is malformed
Error in get_rank_worker(1046): database disk image is malformedError in get_rank_worker(1216): database disk image is malformed
Error in get_rank_worker(11

...

Error in get_rank_worker(8043): database disk image is malformedError in get_rank_worker(8127): file is not a databaseError in get_rank_worker(8211): database disk image is malformed
Error in get_rank_worker(8001): database disk image is malformed
Error in get_rank_worker(8184): database disk image is malformed
Error in get_rank_worker(8088): 'y1'

Error in get_rank_worker(8212): database disk image is malformedError in get_rank_worker(8045): database disk image is malformed
Error in get_rank_worker(8001): database disk image is malformed


Error in get_rank_worker(8046): database disk image is malformedError in get_rank_worker(8001): database disk image is malformed

Error in get_rank_worker(8213): database disk image is malformedError in get_rank_worker(8048): database disk image is malformedError in get_rank_worker(8001): database disk image is malformed


Error in get_rank_worker(8214): database disk image is malformedError in get_rank_worker(8050): database disk image is malformed

Rank phase (fast):  90%|█████████████████████████████████████████████████████████████████████████████▍        | 9/10 [00:05<00:00,  1.51it/s]

Error in get_rank_worker(9169): database disk image is malformedError in get_rank_worker(9127): database disk image is malformedError in get_rank_worker(9044): database disk image is malformedError in get_rank_worker(9085): database disk image is malformedError in get_rank_worker(9003): file is not a database




Error in get_rank_worker(9170): database disk image is malformedError in get_rank_worker(9086): file is not a databaseError in get_rank_worker(9045): file is not a database


Error in get_rank_worker(9215): database disk image is malformedError in get_rank_worker(9171): database disk image is malformedError in get_rank_worker(9088): database disk image is malformedError in get_rank_worker(9046): file is not a database



Error in get_rank_worker(9218): database disk image is malformedError in get_rank_worker(9172): database disk image is malformedError in get_rank_worker(9089): database disk image is malformed


Error in get_rank_worker(9173): database disk image is malformedE

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

BSD phase:   6%|████▉                                                                                   | 1008/17931 [00:12<03:23, 83.33it/s]


KeyboardInterrupt: 